### image processing

imports and paths

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

input paths

In [2]:
MATCHED_V1V2_PATH = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/vent_window_with_best_cxr_V1V2.csv"
OUT_MANIFEST_PATH = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/v1v2_cxr_manifest_within24h.csv"
OUT_EMB_PATH = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/v1v2_cxr_embeddings.csv"

build a clean image manifest

In [5]:
import pandas as pd

matched_v1v2 = pd.read_csv(MATCHED_V1V2_PATH)

# Start from full matched table
base = matched_v1v2.copy()

# Standardize within_24h
base["within_24h"] = base["within_24h"].astype(str).str.lower().map({
    "true": True,
    "false": False
})

# Common filtering shared by both options
base = base[base["within_24h"] == True].copy()
base = base[base["dicom_path"].notna()].copy()

print("After common filters (within_24h + dicom_path):", len(base))

# -----------------------------
# Option A: no ViewPosition filter
# -----------------------------
manifest_a = base.copy()
manifest_a = manifest_a.drop_duplicates()

OUT_MANIFEST_PATH_A = OUT_MANIFEST_PATH.replace(".csv", "_optionA_noViewFilter.csv")
manifest_a.to_csv(OUT_MANIFEST_PATH_A, index=False)

print("\nOption A saved:", OUT_MANIFEST_PATH_A)
print("Option A rows:", len(manifest_a))
print("Option A columns:", manifest_a.shape[1])

# -----------------------------
# Option B: relaxed AP/PA filter
# Keeps AP, PA, AP SUPINE, AP PORTABLE, etc.
# -----------------------------
manifest_b = base.copy()

if "ViewPosition" in manifest_b.columns:
    manifest_b = manifest_b[
        manifest_b["ViewPosition"].fillna("").str.contains("AP|PA", case=False, regex=True)
    ].copy()
else:
    print("\nWarning: ViewPosition column not found, so Option B = Option A")

manifest_b = manifest_b.drop_duplicates()

OUT_MANIFEST_PATH_B = OUT_MANIFEST_PATH.replace(".csv", "_optionB_relaxedAPPA.csv")
manifest_b.to_csv(OUT_MANIFEST_PATH_B, index=False)

print("\nOption B saved:", OUT_MANIFEST_PATH_B)
print("Option B rows:", len(manifest_b))
print("Option B columns:", manifest_b.shape[1])

# -----------------------------
# Optional: inspect ViewPosition distribution
# -----------------------------
if "ViewPosition" in base.columns:
    print("\nTop ViewPosition values before Option B filter:")
    print(base["ViewPosition"].value_counts(dropna=False).head(20))

After common filters (within_24h + dicom_path): 369

Option A saved: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/v1v2_cxr_manifest_within24h_optionA_noViewFilter.csv
Option A rows: 369
Option A columns: 1859

Option B saved: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/v1v2_cxr_manifest_within24h_optionB_relaxedAPPA.csv
Option B rows: 34
Option B columns: 1859

Top ViewPosition values before Option B filter:
ViewPosition
NaN    335
AP      34
Name: count, dtype: int64


DICOM loading and preprocessing helpers

In [6]:
def load_dicom_image(dicom_path):
    """
    Load a DICOM image and return a normalized float32 numpy array in [0, 1].
    """
    ds = pydicom.dcmread(dicom_path)

    # Pixel data
    img = ds.pixel_array

    # Apply VOI LUT if available
    try:
        img = apply_voi_lut(img, ds)
    except Exception:
        pass

    img = img.astype(np.float32)

    # Invert if MONOCHROME1
    photometric = getattr(ds, "PhotometricInterpretation", "")
    if photometric == "MONOCHROME1":
        img = img.max() - img

    # Normalize to [0, 1]
    img = img - img.min()
    max_val = img.max()
    if max_val > 0:
        img = img / max_val

    return img


def preprocess_image_array(img, image_size=224, to_3ch=True):
    """
    Resize and convert to model-ready format.
    Returns CHW float32 array.
    """
    img_uint8 = (img * 255.0).clip(0, 255).astype(np.uint8)
    pil = Image.fromarray(img_uint8).convert("L")
    pil = pil.resize((image_size, image_size))

    arr = np.array(pil, dtype=np.float32) / 255.0

    if to_3ch:
        arr = np.stack([arr, arr, arr], axis=0)  # CHW
    else:
        arr = arr[None, :, :]  # 1 x H x W

    return arr

quick smoke test on one file

In [10]:
test_row = manifest_a.iloc[0]
test_path = test_row["dicom_path"]

img = load_dicom_image(test_path)
x = preprocess_image_array(img, image_size=224, to_3ch=True)

print("Original shape:", img.shape)
print("Processed shape:", x.shape)
print("Min/max:", x.min(), x.max())

Original shape: (1625, 1741)
Processed shape: (3, 224, 224)
Min/max: 0.0 0.91764706


dataset class

In [11]:
class CXRDataset(Dataset):
    def __init__(self, manifest_df, image_size=224, to_3ch=True):
        self.df = manifest_df.reset_index(drop=True).copy()
        self.image_size = image_size
        self.to_3ch = to_3ch

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dicom_path = row["dicom_path"]

        img = load_dicom_image(dicom_path)
        img = preprocess_image_array(img, image_size=self.image_size, to_3ch=self.to_3ch)

        sample = {
            "image": torch.tensor(img, dtype=torch.float32),
            "MRN": row["MRN"],
            "ACC": row["ACC"] if "ACC" in row.index else row["AccessionNumber"],
            "dicom_path": dicom_path,
        }
        return sample

build a pretrained encoder

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
feature_extractor = nn.Sequential(*list(backbone.children())[:-1]).to(device)
feature_extractor.eval()

Using device: cpu


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


run embedding extraction

In [13]:
dataset = CXRDataset(manifest_a, image_size=224, to_3ch=True)
loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=4)

all_mrn = []
all_acc = []
all_path = []
all_emb = []

with torch.no_grad():
    for batch in loader:
        images = batch["image"].to(device)  # [B, 3, 224, 224]
        feat = feature_extractor(images)    # [B, 2048, 1, 1]
        feat = feat.flatten(1).cpu().numpy()

        all_emb.append(feat)
        all_mrn.extend(batch["MRN"])
        all_acc.extend(batch["ACC"])
        all_path.extend(batch["dicom_path"])

emb = np.concatenate(all_emb, axis=0)
print("Embedding shape:", emb.shape)

/home/liuwent/.conda/envs/mamba/lib/python3.10/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Embedding shape: (369, 2048)


save embeddings

In [14]:
emb_df = pd.DataFrame(emb, columns=[f"cxr_emb_{i}" for i in range(emb.shape[1])])
emb_df.insert(0, "dicom_path", all_path)
emb_df.insert(0, "ACC", all_acc)
emb_df.insert(0, "MRN", all_mrn)

emb_df.to_csv(OUT_EMB_PATH, index=False)

print("Saved embeddings:", OUT_EMB_PATH)
print(emb_df.shape)

Saved embeddings: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/v1v2_cxr_embeddings.csv
(369, 2051)


### Merge everything

In [16]:
import pandas as pd
import numpy as np

def mrn_9(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    # remove tensor(...)
    s = s.str.replace(r"^tensor\((.*)\)$", r"\1", regex=True)
    # remove trailing .0
    s = s.str.replace(r"\.0$", "", regex=True)
    # keep digits only
    s = s.str.replace(r"\D", "", regex=True)
    s = s.replace("", pd.NA)
    s = s.str.zfill(9)
    return s

def clean_acc(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.str.replace(r"^tensor\((.*)\)$", r"\1", regex=True)
    s = s.str.strip()
    return s

# Load
vent_df = matched_v1v2.copy()
emb_df = pd.read_csv(OUT_EMB_PATH)

# Standardize keys
vent_df["MRN"] = mrn_9(vent_df["MRN"])
emb_df["MRN"] = mrn_9(emb_df["MRN"])

vent_df["ACC"] = clean_acc(vent_df["ACC"])
emb_df["ACC"] = clean_acc(emb_df["ACC"])

print("vent MRN dtype:", vent_df["MRN"].dtype)
print("emb  MRN dtype:", emb_df["MRN"].dtype)
print("vent ACC dtype:", vent_df["ACC"].dtype)
print("emb  ACC dtype:", emb_df["ACC"].dtype)

print("\nSample vent MRN:", vent_df["MRN"].head().tolist())
print("Sample emb  MRN:", emb_df["MRN"].head().tolist())
print("Sample vent ACC:", vent_df["ACC"].head().tolist())
print("Sample emb  ACC:", emb_df["ACC"].head().tolist())

# Optional: check overlap before merging
print("\nOverlapping MRN:", len(set(vent_df["MRN"].dropna()) & set(emb_df["MRN"].dropna())))
print("Overlapping ACC:", len(set(vent_df["ACC"].dropna()) & set(emb_df["ACC"].dropna())))

# Merge
df_multi = vent_df.merge(
    emb_df,
    on=["MRN", "ACC"],
    how="inner"
)

print("\nFinal multimodal shape:", df_multi.shape)

vent MRN dtype: string
emb  MRN dtype: string
vent ACC dtype: string
emb  ACC dtype: string

Sample vent MRN: ['101643395', '101265981', '101365299', '101649549', '101415800']
Sample emb  MRN: ['101643395', '101265981', '101365299', '101649549', '101415800']
Sample vent ACC: ['MMED40241253', 'MMED39590635', 'MMED40395056', 'MMED40203801', 'MMED40120008']
Sample emb  ACC: ['MMED40241253', 'MMED39590635', 'MMED40395056', 'MMED40203801', 'MMED40120008']

Overlapping MRN: 282
Overlapping ACC: 337

Final multimodal shape: (443, 3908)


In [17]:
key_cols = [c for c in ["MRN", "1st_Time_Start", "ACC"] if c in df_multi.columns]

dup_counts = df_multi.duplicated(subset=key_cols).sum()
print("Duplicate rows by key:", dup_counts)
print("Key columns used:", key_cols)

Duplicate rows by key: 67
Key columns used: ['MRN', '1st_Time_Start', 'ACC']


In [18]:
dups = df_multi[df_multi.duplicated(subset=key_cols, keep=False)].copy()
print(dups[key_cols].head(20))

          MRN           1st_Time_Start           ACC
1   101265981  2024-05-20 17:22:37.000  MMED39590635
2   101265981  2024-05-20 17:22:37.000  MMED39590635
8   101589810  2024-01-27 10:15:08.000  MMED39140422
9   101589810  2024-01-27 10:15:08.000  MMED39140422
17  100160037  2024-04-13 07:06:00.049  MMED39445576
18  100160037  2024-04-13 07:06:00.049  MMED39445576
22  040345619  2024-02-26 00:00:18.000  MMED39253761
23  040345619  2024-02-26 00:00:18.000  MMED39253761
24  040345619  2025-02-12 15:59:13.000  MMED39253761
25  040345619  2025-02-12 15:59:13.000  MMED39253761
27  039775438  2024-02-01 06:05:00.035  MMED39154216
28  039775438  2024-02-01 06:05:00.035  MMED39154216
34  101363585  2024-05-01 08:22:01.000  MMED39515794
35  101363585  2024-05-01 08:22:01.000  MMED39515794
36  045575238  2024-01-13 23:25:58.000  MMED39086290
37  045575238  2024-01-13 23:25:58.000  MMED39086290
40  100648830  2024-02-07 22:28:00.011  MMED39185886
41  100648830  2024-02-07 22:28:00.011  MMED39

In [19]:
df_multi = df_multi.drop_duplicates(subset=key_cols).copy()
print(df_multi.shape)

(376, 3908)


In [ ]:
OUT_MULTI_PATH = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/v1v2_multimodal_dataset.csv"

df_multi.to_csv(OUT_MULTI_PATH, index=False)

print("Saved df_multi to:", OUT_MULTI_PATH)
print("Shape:", df_multi.shape)

Saved df_multi to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/v1v2_multimodal_dataset.csv
Shape: (376, 3908)
